<a href="https://colab.research.google.com/github/mudassir112256/ai-video-clipper/blob/main/chapter_appendix-tools-for-deep-learning/jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
# ==========================================
# STEP 1: INSTALL DEPENDENCIES
# ==========================================
import datetime
import json
import os
import subprocess
import cv2
import yt_dlp
from google.colab import drive, userdata

!pip install -q yt-dlp faster-whisper ffmpeg-python google-genai opencv-python-headless
!apt-get install -y ffmpeg

# ==========================================
# CONFIGURATION
# ==========================================
VIDEO_URL = "https://youtu.be/7RaC2nKBqv4?si=_JMt_H897Jl6wbGz"  # Replace with your YouTube URL
WATERMARK_TEXT = "@neterogoat"                         # Your social handle
ENABLE_GDRIVE_BACKUP = True                                 # Auto-save outputs to Google Drive

# Caption Highlight Colors (ASS Format - &H00BBGGRR)
HIGHLIGHT_COLOR = "&H0000FFFF&"  # Bright Yellow
DEFAULT_COLOR   = "&H00FFFFFF&"  # Clean White

# Load API key safely from Colab Secrets (🔑)
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("🔑 Gemini API key loaded from Colab Secrets!")
except Exception as e:
    print("⚠️ Could not load GEMINI_API_KEY from Secrets. Ensure it is set in the Secrets tab (🔑).")

# ==========================================
# STEP 2: DOWNLOAD VIDEO
# ==========================================
print("\n📥 Step 1/6: Downloading video...")
ydl_opts = {
    'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]',
    'outtmpl': 'input_video.mp4',
    'extractor_args': {
        'youtube': {
            'player_client': ['android', 'ios'],
        }
    },
}
with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download([VIDEO_URL])
print("✅ Video downloaded!\n")

# ==========================================
# STEP 3: FACE TRACKING CENTER COMPUTATION
# ==========================================
def detect_speaker_center(video_path):
    print("👤 Step 2/6: Running OpenCV face tracking...")
    cap = cv2.VideoCapture(video_path)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    x_positions = []
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_count > 1500: # Sample first ~50 seconds
            break
        if frame_count % 15 == 0: # Sample every 15th frame
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.1, 4)
            for (x, y, w, h) in faces:
                x_positions.append(x + w // 2)
        frame_count += 1

    cap.release()
    if x_positions:
        avg_x = int(sum(x_positions) / len(x_positions))
        print(f"✅ Speaker center found at X coordinate: {avg_x}px\n")
        return avg_x
    print("⚠️ No face detected; using default center crop.\n")
    return None

speaker_x = detect_speaker_center("input_video.mp4")

# ==========================================
# STEP 4: FAST WHISPER TRANSCRIPTION (FASTER-WHISPER)
# ==========================================
print("⚡ Step 3/6: Transcribing audio with Faster-Whisper (GPU)...")
from faster_whisper import WhisperModel

# Run on GPU with float16 precision for max performance
model = WhisperModel("small", device="cuda", compute_type="float16")

segments_generator, info = model.transcribe("input_video.mp4", word_timestamps=True)

# Format outputs for down-stream processing
result = {"segments": []}
transcript_text = ""

for seg in segments_generator:
    words = []
    if seg.words:
        for w in seg.words:
            words.append({"word": w.word, "start": w.start, "end": w.end})

    seg_dict = {
        "start": seg.start,
        "end": seg.end,
        "text": seg.text,
        "words": words
    }
    result["segments"].append(seg_dict)
    transcript_text += f"[{seg.start:.2f}s - {seg.end:.2f}s]: {seg.text}\n"

print("✅ Accelerated transcription complete!\n")

# ==========================================
# STEP 5: GENERATE ANIMATED SUBTITLES (.ASS)
# ==========================================
def format_ass_time(seconds):
    td = datetime.timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    centisecs = int((seconds - int(seconds)) * 100)
    return f"{hours:01d}:{minutes:02d}:{secs:02d}.{centisecs:02d}"

def create_animated_ass(segments, output_ass="animated_subs.ass"):
    header = f"""[Script Info]
ScriptType: v4.00+
PlayResX: 1080
PlayResY: 1920

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Highlight,Arial,65,{DEFAULT_COLOR},&H0000FFFF,&H00000000,&H80000000,1,0,0,0,100,100,0,0,1,4,0,2,50,50,350,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
    events = []
    for seg in segments:
        words = seg.get("words", [])
        if not words:
            continue
        chunk_size = 4
        for i in range(0, len(words), chunk_size):
            chunk = words[i:i + chunk_size]
            for w_idx, active_word in enumerate(chunk):
                w_start = format_ass_time(active_word["start"])
                w_end = format_ass_time(active_word["end"])
                formatted_text = ""
                for idx, word_info in enumerate(chunk):
                    text_clean = word_info["word"].strip().upper()
                    if idx == w_idx:
                        # Fix for SyntaxError: f-string: single '}' is not allowed and SyntaxWarning: invalid escape sequence '\c'
                        formatted_text += f"{{\\c{HIGHLIGHT_COLOR}}}" + f"{text_clean} "
                    else:
                        # Fix for SyntaxError: f-string: single '}' is not allowed and SyntaxWarning: invalid escape sequence '\c'
                        formatted_text += f"{{\\c{DEFAULT_COLOR}}}" + f"{text_clean} "
                events.append(f"Dialogue: 0,{w_start},{w_end},Highlight,,0,0,0,,{formatted_text.strip()}")

    with open(output_ass, "w", encoding="utf-8") as f:
        f.write(header + "\n".join(events))

print("📝 Step 4/6: Building word-highlight ASS subtitles...")
create_animated_ass(result["segments"], "animated_subs.ass")
print("✅ Subtitles built!\n")

# ==========================================
# STEP 6: GEMINI VIRAL & TITLE SELECTION
# ==========================================
print("🤖 Step 5/6: Finding viral moments and titles with Gemini AI...")
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)
prompt = f"Analyze transcript. Pick top 3 viral moments (15-60s). Provide a short catchy viral title for each.\n\nTranscript:\n{transcript_text}"

response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema={
            "type": "ARRAY",
            "items": {
                "type": "OBJECT",
                "properties": {
                    "start": {"type": "NUMBER"},
                    "end": {"type": "NUMBER"},
                    "title": {"type": "STRING"},
                    "reason": {"type": "STRING"}
                },
                "required": ["start", "end", "title", "reason"]
            }
        }
    )
)

clips_data = json.loads(response.text)
print(f"✅ Gemini selected {len(clips_data)} clips with titles!\n")

# ==========================================
# STEP 7: EXPORT FFMPEG VERTICAL CLIPS WITH WATERMARK
# ==========================================
def export_short(input_video, ass_file, start, end, out_name, crop_x=None):
    duration = end - start

    if crop_x:
        # Fix for SyntaxWarning: invalid escape sequence '\,'
        base_crop = f"crop=ih*(9/16):ih:min(max(0\\,{crop_x}-((ih*(9/16))/2))\\,iw-(ih*(9/16))):0,scale=1080:1920"
    else:
        base_crop = "crop=ih*(9/16):ih:(iw-out_w)/2:0,scale=1080:1920"

    watermark_filter = (
        f"drawtext=text='{WATERMARK_TEXT}':fontsize=48:fontcolor=white:"
        f"x=(w-tw)/2:y=120:shadowcolor=black@0.8:shadowx=3:shadowy=3"
    )

    vf_filter = f"{base_crop},{watermark_filter},ass={ass_file}"

    cmd = [
        "ffmpeg", "-y", "-ss", str(start), "-i", input_video,
        "-t", str(duration), "-vf", vf_filter,
        "-c:v", "libx264", "-preset", "fast", "-c:a", "aac", out_name
    ]
    subprocess.run(cmd, check=True)

print("✂️ Step 6/6: Rendering final vertical shorts with watermark...")
for i, clip in enumerate(clips_data):
    clean_title = "".join(c for c in clip['title'] if c.isalnum() or c in (' ', '_')).rstrip().replace(" ", "_")
    out_file = f"viral_{i+1}_{clean_title}.mp4"
    print(f"Rendering: {clip['title']} ({clip['start']}s - {clip['end']}s)")
    export_short("input_video.mp4", "animated_subs.ass", clip['start'], clip['end'], out_file, speaker_x)

# Google Drive Auto-Backup
if ENABLE_GDRIVE_BACKUP:
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/AI_Shorts_Output', exist_ok=True)
    !cp viral_*.mp4 /content/drive/MyDrive/AI_Shorts_Output/
    print("\n📁 Copies uploaded to Google Drive: MyDrive/AI_Shorts_Output/")

print("\n🎉 ALL DONE! Check the left folder icon (📁) or Google Drive for your rendered clips!")


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 57 not upgraded.
🔑 Gemini API key loaded from Colab Secrets!

📥 Step 1/6: Downloading video...
[youtube] Extracting URL: https://youtu.be/7RaC2nKBqv4?si=_JMt_H897Jl6wbGz
[youtube] 7RaC2nKBqv4: Downloading webpage
[youtube] 7RaC2nKBqv4: Downloading android player API JSON
[youtube] 7RaC2nKBqv4: Downloading ios player API JSON


ERROR: [youtube] 7RaC2nKBqv4: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies


DownloadError: ERROR: [youtube] 7RaC2nKBqv4: Sign in to confirm you’re not a bot. Use --cookies-from-browser or --cookies for the authentication. See  https://github.com/yt-dlp/yt-dlp/wiki/FAQ#how-do-i-pass-cookies-to-yt-dlp  for how to manually pass cookies. Also see  https://github.com/yt-dlp/yt-dlp/wiki/Extractors#exporting-youtube-cookies  for tips on effectively exporting YouTube cookies